<a href="https://colab.research.google.com/github/night-marionette/Pneumonia-Triage/blob/main/01_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
OUT_DIR = '/content/drive/MyDrive/PneumoniaTriage'
os.makedirs(OUT_DIR, exist_ok=True)
print("Folder ready:", OUT_DIR)

Folder ready: /content/drive/MyDrive/PneumoniaTriage


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [ ]:
import os
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from collections import defaultdict
from PIL import ImageOps
from sklearn.metrics import roc_auc_score

# 1. FIX SEEDS FOR REPRODUCIBILITY
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

# 2. SETUP PATHS & DEVICE (Auto-finds dataset safely)
if 'path' in globals() and os.path.exists(os.path.join(path, 'chest_xray')):
    DATA_DIR = os.path.join(path, 'chest_xray')
else:
    DATA_DIR = '/kaggle/input/chest-xray-pneumonia/chest_xray'

OUT_DIR = '/content/drive/MyDrive/PneumoniaTriage'
os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Using dataset:", DATA_DIR)

# 3. PAD TO SQUARE SO X-RAYS DON'T GET SQUISHED
class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)
        pad_left = (max_wh - w) // 2
        pad_top = (max_wh - h) // 2
        pad_right = max_wh - w - pad_left
        pad_bottom = max_wh - h - pad_top
        return ImageOps.expand(image, (pad_left, pad_top, pad_right, pad_bottom), fill=(0, 0, 0))

# 4. TRANSFORMS
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    SquarePad(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    SquarePad(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# 5. LOAD DATA WITH PATIENT-LEVEL SPLIT (NO DATA LEAKAGE)
train_folder = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_tf)
val_folder   = datasets.ImageFolder(f'{DATA_DIR}/train', transform=eval_tf)
test_ds      = datasets.ImageFolder(f'{DATA_DIR}/test', transform=eval_tf)

def extract_patient_id(img_path):
    fname = os.path.basename(img_path)
    match = re.match(r'^(person\d+|IM-\d+|NORMAL2-IM-\d+)', fname)
    return match.group(1) if match else fname

patient_to_indices = defaultdict(list)
for idx, (img_path, _) in enumerate(train_folder.samples):
    pid = extract_patient_id(img_path)
    patient_to_indices[pid].append(idx)

all_patients = sorted(list(patient_to_indices.keys()))
rng = random.Random(42)
rng.shuffle(all_patients)

val_patient_count = int(0.15 * len(all_patients))
val_patients = set(all_patients[:val_patient_count])
train_patients = set(all_patients[val_patient_count:])

train_indices = [idx for pid in train_patients for idx in patient_to_indices[pid]]
val_indices   = [idx for pid in val_patients   for idx in patient_to_indices[pid]]

train_ds = Subset(train_folder, train_indices)
val_ds   = Subset(val_folder, val_indices)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"Classes: {train_folder.classes}")
print(f"Unique Patients: {len(all_patients)} (Train: {len(train_patients)}, Val: {len(val_patients)})")
print(f"Images -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# 6. FIX CLASS IMBALANCE (So AI doesn't over-flag healthy kids)
train_labels = [train_folder.samples[i][1] for i in train_indices]
class_counts = np.bincount(train_labels)
class_weights = len(train_labels) / (2.0 * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class Counts: {class_counts} | Weights Used: {class_weights.round(2)}")

# 7. UPGRADED MODEL (DENSENET-121)
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(model.classifier.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)

# 8. TRAIN LOOP + SAVE BEST EPOCH ONLY
EPOCHS = 6
best_val_auc = 0.0
model_save_path = f'{OUT_DIR}/densenet121_pneumonia.pth'

print(f"\nStarting DenseNet-121 training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0, 0
    val_probs, val_true = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            val_probs.extend(probs.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    val_acc = 100 * correct / total
    val_auc = roc_auc_score(val_true, val_probs)

    saved_msg = ""
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), model_save_path)
        saved_msg = " <-- NEW BEST SAVED!"

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.3f} | Val Acc: {val_acc:.1f}% | Val AUC: {val_auc:.4f}{saved_msg}")

# 9. LOAD BEST WEIGHTS BEFORE TESTING
model.load_state_dict(torch.load(model_save_path))
print(f"\nLoaded best model weights (Best Val AUC: {best_val_auc:.4f})")

# 10. GENERATE FINAL TEST_SCORES.CSV
print("Generating final test scores...")
model.eval()
all_scores = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        all_scores.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

df = pd.DataFrame({
    "id": range(len(all_labels)),
    "true_label": all_labels,
    "score": all_scores
})

save_path = f'{OUT_DIR}/test_scores.csv'
df.to_csv(save_path, index=False)
print(f"SUCCESS! Saved {len(df)} rows to {save_path}")

Using device: cuda
Using dataset: /kaggle/input/chest-xray-pneumonia/chest_xray
Classes: ['NORMAL', 'PNEUMONIA']
Unique Patients: 2846 (Train: 2420, Val: 426)
Images -> Train: 4395 | Val: 821 | Test: 624
Class Counts: [1155 3240] | Weights Used: [1.9  0.68]
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 162MB/s]



Starting DenseNet-121 training for 6 epochs...
Epoch 1/6 | Loss: 0.129 | Val Acc: 98.8% | Val AUC: 0.9990 <-- NEW BEST SAVED!
Epoch 2/6 | Loss: 0.057 | Val Acc: 97.1% | Val AUC: 0.9987
Epoch 3/6 | Loss: 0.036 | Val Acc: 98.4% | Val AUC: 0.9993 <-- NEW BEST SAVED!
Epoch 4/6 | Loss: 0.031 | Val Acc: 97.4% | Val AUC: 0.9993 <-- NEW BEST SAVED!
Epoch 5/6 | Loss: 0.022 | Val Acc: 98.7% | Val AUC: 0.9993
Epoch 6/6 | Loss: 0.014 | Val Acc: 98.5% | Val AUC: 0.9996 <-- NEW BEST SAVED!

Loaded best model weights (Best Val AUC: 0.9996)
Generating final test scores...


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

labels_array = np.array(all_labels)
scores_array = np.array(all_scores)
preds = (scores_array >= 0.5).astype(int)

# 1. TEXT METRICS
print("=== BASELINE (Always guess Pneumonia) ===")
base_preds = np.ones_like(labels_array)
print(f"Accuracy: {(base_preds == labels_array).mean()*100:.1f}%\n")

print("=== DENSENET-121 MODEL EVALUATION (@ 0.5 Threshold) ===")
print(f"Accuracy: {(preds == labels_array).mean()*100:.1f}%")

tp = ((preds == 1) & (labels_array == 1)).sum()
fn = ((preds == 0) & (labels_array == 1)).sum()
fp = ((preds == 1) & (labels_array == 0)).sum()
tn = ((preds == 0) & (labels_array == 0)).sum()

print(f"Pneumonia Recall (Sick Kids Caught): {tp / (tp + fn) * 100:.1f}% | Missed (FN): {fn}")
print(f"False Alarms (Healthy Kids Flagged): {fp} | Specificity: {tn / (tn + fp) * 100:.1f}%\n")

# 2. REPORT CARD CHARTS: CONFUSION MATRIX & ROC-AUC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(labels_array, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NORMAL', 'PNEUMONIA'])
disp.plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title('Confusion Matrix (@ 0.5 Threshold)', fontsize=13, fontweight='bold')

fpr, tpr, _ = roc_curve(labels_array, scores_array)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'DenseNet-121 (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Guess (AUC = 0.50)')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (Healthy Flagged)', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Sick Caught)', fontsize=11)
axes[1].set_title(f'ROC Curve (AUROC: {roc_auc:.4f})', fontsize=13, fontweight='bold')
axes[1].legend(loc="lower right", fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/model_report_card.png', dpi=200)
plt.show()

# 3. DISPLAY MISSED PNEUMONIA CASES
miss_idx = np.where((labels_array == 1) & (preds == 0))[0]
print(f"\nShowing {min(3, len(miss_idx))} missed pneumonia cases...")
if len(miss_idx) > 0:
    fig, axes = plt.subplots(1, min(3, len(miss_idx)), figsize=(10, 3))
    if min(3, len(miss_idx)) == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        idx = miss_idx[i]
        img_path = test_ds.samples[idx][0]
        score = scores_array[idx]
        ax.imshow(Image.open(img_path).convert("L"), cmap="gray")
        ax.set_title(f"Missed Case #{i+1}\nAI Score: {score:.2f}")
        ax.axis("off")
    plt.show()

In [ ]:
!pip install grad-cam -q
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image

# 1. Point GradCAM++ at DenseNet-121's last conv block
model.eval()
target_layers = [model.features.denseblock4]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

# 2. Grab a sick patient from the test set
labels_array = np.array(all_labels)
pneumonia_idx = np.where(labels_array == 1)[0][0]
img_path = test_ds.samples[pneumonia_idx][0]

# 3. Prepare image with EXACT same SquarePad as the model
pil_img = Image.open(img_path).convert('L').convert('RGB')
padded_rgb = SquarePad()(pil_img).resize((224, 224))
rgb_img_float = np.float32(padded_rgb) / 255.0

input_tensor = eval_tf(pil_img).unsqueeze(0).to(device)

# 4. Generate GradCAM++ heatmap for PNEUMONIA (index 1)
targets = [ClassifierOutputTarget(1)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
visualization = show_cam_on_image(rgb_img_float, grayscale_cam, use_rgb=True)

# 5. Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(padded_rgb)
axes[0].set_title("Original X-Ray (Padded)")
axes[0].axis("off")

axes[1].imshow(visualization)
axes[1].set_title("AI Heatmap - GradCAM++ (Red = Looking here)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 1. Find an image in the test set that ACTUALLY has pneumonia
labels_array = np.array(all_labels)
pneumonia_idx = np.where(labels_array == 1)[0][0] # Grabs the first sick patient
img_path = test_ds.samples[pneumonia_idx][0]

# 2. Prepare the image for the model
raw_img = cv2.imread(img_path)
raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
rgb_img = cv2.resize(raw_img, (224, 224))
rgb_img_float = np.float32(rgb_img) / 255

input_tensor = eval_tf(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)

# 3. Generate the heatmap for the "PNEUMONIA" class (index 1)
targets = [ClassifierOutputTarget(1)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
visualization = show_cam_on_image(rgb_img_float, grayscale_cam, use_rgb=True)

# 4. Plot them side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb_img)
axes[0].set_title("Original X-Ray")
axes[0].axis("off")

axes[1].imshow(visualization)
axes[1].set_title("AI Heatmap (Red = Looking here)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install gradio -q
import gradio as gr
from PIL import Image
import torch
import numpy as np
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

model.eval()
target_layers = [model.features.denseblock4]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

def predict_and_explain(image, threshold):
    if image is None:
        return "Please upload an X-ray.", None, None

    # 1. Preprocess with SquarePad (matches training)
    pil_img = image.convert('L').convert('RGB')
    padded_rgb = SquarePad()(pil_img).resize((224, 224))
    rgb_float = np.float32(padded_rgb) / 255.0

    input_tensor = eval_tf(pil_img).unsqueeze(0).to(device)

    # 2. Model Prediction
    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]

    normal_prob = float(probs[0].item())
    pneumonia_prob = float(probs[1].item())

    # 3. Triage Decision
    if pneumonia_prob >= threshold:
        triage_decision = f"🔴 PNEUMONIA - PRIORITY TRIAGE ({pneumonia_prob*100:.1f}%)"
    else:
        triage_decision = f"🟢 NORMAL - STANDARD QUEUE ({normal_prob*100:.1f}%)"

    prob_dict = {
        "Pneumonia": pneumonia_prob,
        "Normal": normal_prob
    }

    # 4. Generate GradCAM++ Heatmap
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(1)])[0, :]
    heatmap_overlay = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)

    return triage_decision, prob_dict, heatmap_overlay

demo = gr.Interface(
    fn=predict_and_explain,
    inputs=[
        gr.Image(type="pil", label="Upload Pediatric Chest X-Ray"),
        gr.Slider(minimum=0.1, maximum=0.99, value=0.5, step=0.05, label="Triage Threshold Cutoff")
    ],
    outputs=[
        gr.Textbox(label="Triage Decision"),
        gr.Label(label="AI Confidence"),
        gr.Image(type="numpy", label="GradCAM++ Lung Heatmap (Red = AI Focus)")
    ],
    title="Pediatric Pneumonia AI Triage & Explainability (DenseNet-121)",
    description="Upload a chest X-ray and adjust the triage slider. The model outputs the priority decision and a GradCAM++ heatmap highlighting suspicious lung areas."
)

demo.launch(share=True)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/PneumoniaTriage/test_scores.csv')
labels_array = df['true_label'].values
scores_array = df['score'].values
print(f"Loaded {len(df)} rows | pneumonia: {(labels_array==1).sum()} | normal: {(labels_array==0).sum()}")

In [ ]:
print(f"{'Thresh':>7} | {'Acc':>6} | {'Recall':>7} | {'FN':>3} | {'FP':>4} | {'Spec':>6}")
print("-" * 55)

for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    p = (scores_array >= t).astype(int)
    tp = ((p==1)&(labels_array==1)).sum(); fn = ((p==0)&(labels_array==1)).sum()
    fp = ((p==1)&(labels_array==0)).sum(); tn = ((p==0)&(labels_array==0)).sum()
    acc  = (p==labels_array).mean()*100
    rec  = tp/(tp+fn)*100
    spec = tn/(tn+fp)*100
    print(f"{t:>7.2f} | {acc:>5.1f}% | {rec:>6.1f}% | {fn:>3} | {fp:>4} | {spec:>5.1f}%")

In [ ]:
import hashlib
h = hashlib.md5(open('/content/drive/MyDrive/PneumoniaTriage/test_scores.csv', 'rb').read()).hexdigest()
print('Hash of YOUR file:', h)